Step 1: Read in, unzip, and look at the first 5 rows of the circRNA-seq



In [ ]:
!wget https://ftp.ncbi.nlm.nih.gov/geo/series/GSE133nnn/GSE133319/suppl/GSE133319%5FmiRNA%2Dexp%2Etxt%2Egz #miRNA




--2026-05-14 23:05:40--  https://ftp.ncbi.nlm.nih.gov/geo/series/GSE133nnn/GSE133319/suppl/GSE133319%5FmiRNA%2Dexp%2Etxt%2Egz
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.11, 130.14.250.12, 2607:f220:41e:250::11, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.11|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 22871 (22K) [application/x-gzip]
Saving to: ‘GSE133319_miRNA-exp.txt.gz.1’

GSE133319_miRNA-exp 100%[===================>]  22.33K  --.-KB/s    in 0.01s   

2026-05-14 23:05:40 (1.69 MB/s) - ‘GSE133319_miRNA-exp.txt.gz.1’ saved [22871/22871]



In [ ]:
!gzip -d GSE133319_miRNA-exp.txt.gz


gzip: GSE133319_miRNA-exp.txt already exists; do you wish to overwrite (y or n)? 

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("GSE133319_miRNA-exp.txt", sep="\t")

df = df.set_index("tracking_id")

df.head()

Trying the analysis again with PyDeSEQ2:

LLM prompt: using Deseq2 in python, perform a DEG analysis comparing control groups (con3, con4, and con5) versus infected groups (HV2, HV4, HV6). Obtain the log2FC, the FDRs, and filter based on |log2FC| > 1 and FDR < 0.05.

In [ ]:
!pip install pydeseq2
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
import pandas as pd

# PyDESeq2 needs RAW counts (integers), not log-transformed RPM
counts = df.T

counts = counts.astype(int)

# Metadata dataframe
metadata = pd.DataFrame({
    'condition': ['control', 'control', 'control', 'infected', 'infected', 'infected']
}, index=counts.index)

# Run DESeq2
dds = DeseqDataSet(counts=counts, metadata=metadata, design_factors="condition")
dds.deseq2()

# Extract results
stat_res = DeseqStats(dds, contrast=["condition", "infected", "control"])
stat_res.summary()

results_deseq = stat_res.results_df
sig = results_deseq[(results_deseq['padj'] < 0.05) & (results_deseq['log2FoldChange'].abs() > 1)]
print(f"Significant DEGs: {len(sig)}")

sig.head()

In [ ]:
sig.head()

In [ ]:
sig.shape #Gives us 97 DEGs which is a lot closer to the 70 DEGs that the authors identified...

Make volcano

In [ ]:
pvs   = results_deseq['pvalue'].values
logFC = results_deseq['log2FoldChange'].values

print("Total missing p-values:", np.isnan(pvs).sum())

Remove those 0/NA p-values

In [ ]:
valid  = ~np.isnan(pvs)
pvs_valid   = pvs[valid]
logFC_valid = logFC[valid]

In [ ]:
fdrs_valid = results_deseq['padj'].values[valid]

# P-value cutoff at FDR=0.05
p_fdr005     = pvs_valid[fdrs_valid < 0.05].max()
p_cutoff     = p_fdr005
logFC_cutoff = 1
print("P-value at FDR=0.05:", p_fdr005)

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(logFC_valid, -np.log10(pvs_valid), alpha=0.4, c='grey', s=10)
plt.xlabel('log2(Fold Change)')
plt.ylabel('-log10(p-value)')
plt.axhline(-np.log10(p_cutoff), color='r', linestyle='--', label='FDR=0.05')
plt.axvline( logFC_cutoff,       color='g', linestyle='--', label='log2(Fold Change)=1')
plt.axvline(-logFC_cutoff,       color='g', linestyle='--')
plt.fill_between([-6, -1], [-np.log10(p_cutoff), -np.log10(p_cutoff)], [25, 25], alpha=0.2, color='blue',   label='Down')
plt.fill_between([ 1,  6], [-np.log10(p_cutoff), -np.log10(p_cutoff)], [25, 25], alpha=0.2, color='orange', label='Up')
plt.xlim(-6, 6)
plt.ylim(-0.05, 25)
plt.legend()
plt.title('Volcano Plot: HV vs Control circRNA Expression (DESeq2)')
plt.tight_layout()
plt.show()



In [ ]:
#LLM prompt: tell me the names of the DEGs, and then tell me the names of the upregulated and downregulated ones
print(f"Total significant circRNAs: {len(sig)}")
print(f"Upregulated   (log2FC >  1): {len(sig[sig['log2FoldChange'] >  1])}")
print(f"Downregulated (log2FC < -1): {len(sig[sig['log2FoldChange'] < -1])}")

#Pretty good, the authors identified 70 DEGs, 65 up and 5 down so this is somewhat consistent. Only difference is we used pydeseq2 and they used edgeR.

TF analysis


In [ ]:
!pip install gseapy -q

import pandas as pd
import gseapy as gp
from gseapy import barplot, dotplot
import matplotlib.pyplot as plt


In [ ]:
deg_mirna = sig.index.tolist()

print(f"Total miRNA DEGs: {len(deg_mirna)}")
print(deg_mirna[:10])

Need to convert these to real gene names somehow --> LLM: converse hsa-miR form to actual gene names for the deg_miRNAs

In [ ]:
tf_results_mirna = gp.enrichr(
    gene_list=deg_mirna,
    gene_sets=[
        'ChEA_2022',
        'ENCODE_TF_ChIP-seq_2015',
        'TF_Perturbations_Followed_by_Expression'
    ],
    organism='human',
    outdir=None
)

tf_results_mirna.results.head(5)

Nothing popping up, so we will read in the miRNA database csv and try to make a graph

In [ ]:
mirtarbase = pd.read_csv('mir2tf.csv')

# Count how many times each TF appears
tf_counts = mirtarbase['Target'].value_counts()

print(tf_counts.head(20))

In [ ]:
tf_counts.head(15).plot(
    kind='barh',
    figsize=(8, 6),
    color='steelblue'
)
plt.xlabel('Number of miRNAs targeting this TF')
plt.title('Most Frequently Targeted TFs by DEG miRNAs')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
#LLM Prompt: write me code in Python that takes the miRNA data as the input and outputs me a list of the top 10 TFs arranged from lowest adj. P value to highest.

top_tfs = mirtarbase['Target'].value_counts().index.tolist()

print(top_tfs)


['RELA', 'NFKB1', 'PBX2', 'MYC', 'HIF1A', 'HNF4A', 'SPI1', 'JUN', 'EGR1', 'MYCN', 'BRCA1', 'MITF', 'SRF', 'TP53', 'STAT3', 'ZBTB16', 'TWIST1', 'SOX2', 'PPARA', 'FOSL1', 'REST', 'CEBPA', 'EGR3', 'BANP', 'ATOH8', 'ATF3', 'BRD4', 'ILF3', 'ILF2', 'IL1B', 'HOXD10', 'HOXB7', 'FOXP3', 'GATA2', 'HDAC1', 'ESR1', 'ETS1', 'EPAS1', 'EZH2', 'CBFB', 'CTNNB1', 'EED', 'NR3C2', 'RELB', 'RBPJ', 'PSMD9', 'RBBP5', 'KMT2A', 'NFE2L2', 'NFKB2', 'RUNX1', 'SRSF1', 'RUNX1T1', 'RUNX3', 'SNAI2', 'TLR2', 'TGFB1', 'TCF4', 'TNFSF12']
